In [1]:
!pip install transformers scikit-learn

In [2]:
import pickle
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
model_path = "/content/drive/My Drive/INTERNSHIP/plant_disease_text_model"
encoder_path = "/content/drive/My Drive/INTERNSHIP/label_encoder.pkl"

In [5]:
# load the trained DistilBERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.eval()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [6]:
# load the label encoder
with open(encoder_path, "rb") as f:
    encoder = pickle.load(f)

In [7]:
# define a function that predicts the plant disease based on text
def predict_disease(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()

    disease = encoder.inverse_transform([predicted_class])[0]

    return disease

In [8]:
# test the model with unseen text descriptions
text1 = "Tomato leaf with yellow spots and curling edges"
text2 = "Potato leaf showing dark brown patches with yellow halo"
text3 = "Healthy green leaf with no visible disease"

print("Prediction 1:", predict_disease(text1))
print("Prediction 2:", predict_disease(text2))
print("Prediction 3:", predict_disease(text3))

Prediction 1: Tomato YellowLeaf Curl Virus
Prediction 2: Potato Late blight
Prediction 3: Potato healthy
